In [1]:
# W4D4: FastAPI Model Serving Endpoint
# One-cell workflow: train and persist a pipeline, serve typed predictions, and capture API evidence.
from pathlib import Path
import json
import hashlib
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import numpy as np
import pandas as pd
import sklearn
import fastapi
import mlflow
from fastapi import FastAPI, HTTPException
from starlette.testclient import TestClient
from pydantic import BaseModel, Field
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

SEED = 42
# Resolve output location whether this cell runs from the repo root or week4_Notebooks.
ROOT = Path.cwd().parent if Path.cwd().name == "week4_Notebooks" else Path.cwd()
OUT = ROOT / "outputs" / "w4d4_fastapi_serving"
OUT.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT / "breast_cancer_pipeline.joblib"

# Train a reproducible model and keep preprocessing attached to the estimator.
X_all, y_all = load_breast_cancer(return_X_y=True)
CLASS_NAMES = ("malignant", "benign")
FEATURE_COUNT = X_all.shape[1]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, stratify=y_all, random_state=SEED
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED))
model.fit(X_train, y_train)
joblib.dump(model, MODEL_PATH, compress=3)

# Load from disk as a real serving process would, then define a stable request/response contract.
served_model = joblib.load(MODEL_PATH)
class PredictionRequest(BaseModel):
    features: list[float] = Field(..., description="Ordered numeric model features")
class PredictionResponse(BaseModel):
    prediction: int
    label: str
    probability: float
app = FastAPI(title="W4D4 Breast Cancer Model API", version="1.0.0")
@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_PATH.name}
@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    if len(request.features) != FEATURE_COUNT:
        raise HTTPException(status_code=422, detail=f"Expected {FEATURE_COUNT} features; received {len(request.features)}")
    values = np.asarray(request.features, dtype=float).reshape(1, -1)
    prediction = int(served_model.predict(values)[0])
    probability = float(served_model.predict_proba(values)[0, 1])
    return PredictionResponse(prediction=prediction, label=CLASS_NAMES[prediction], probability=probability)

# Exercise the HTTP contract without binding a network port; this is deterministic and CI-friendly.
client = TestClient(app)
health_response = client.get("/health")
assert health_response.status_code == 200 and health_response.json()["status"] == "ok"
sample = X_test[0].tolist()
response = client.post("/predict", json={"features": sample})
assert response.status_code == 200, response.text
payload = response.json()
assert payload["prediction"] == int(served_model.predict(X_test[:1])[0])
assert 0.0 <= payload["probability"] <= 1.0
invalid = client.post("/predict", json={"features": sample[:-1]})
assert invalid.status_code == 422

# Record holdout quality and an isolated SQLite-backed MLflow run for model lineage.
y_pred = served_model.predict(X_test)
y_prob = served_model.predict_proba(X_test)[:, 1]
metrics = {"accuracy": float(accuracy_score(y_test, y_pred)),
           "precision": float(precision_score(y_test, y_pred)),
           "recall": float(recall_score(y_test, y_pred)),
           "roc_auc": float(roc_auc_score(y_test, y_prob))}
mlflow.set_tracking_uri(f"sqlite:///{(OUT / 'mlflow.db').as_posix()}")
with mlflow.start_run(run_name="w4d4-fastapi-serving") as run:
    mlflow.log_params({"model": "StandardScaler + LogisticRegression", "seed": SEED,
                       "test_size": 0.25, "api_version": app.version})
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(str(MODEL_PATH), artifact_path="model")

manifest = {"api": {"title": app.title, "version": app.version, "routes": ["GET /health", "POST /predict"]},
            "model": MODEL_PATH.name, "model_sha256": hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
            "input_feature_count": int(FEATURE_COUNT), "classes": list(CLASS_NAMES),
            "versions": {"scikit_learn": sklearn.__version__, "fastapi": fastapi.__version__, "mlflow": mlflow.__version__},
            "metrics": metrics, "api_checks": {"health_status": health_response.status_code,
            "prediction_status": response.status_code, "wrong_feature_count_status": invalid.status_code,
            "sample_prediction": payload}, "mlflow_run_id": run.info.run_id}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
pd.DataFrame([{"check": "health", "status": health_response.status_code},
              {"check": "valid_prediction", "status": response.status_code},
              {"check": "wrong_feature_count", "status": invalid.status_code}]).to_csv(OUT / "api_checks.csv", index=False)
print("API checks:\n", pd.read_csv(OUT / "api_checks.csv").to_string(index=False))
print("Holdout metrics:", json.dumps(metrics, indent=2))
print("MLflow run:", run.info.run_id)
print("Evidence:", OUT.resolve())
print("Self-review: model persisted and reloaded; typed endpoints exercised; malformed input rejected; metrics, versions, hash, and run ID recorded.")


C:\Users\Renuk\AppData\Local\Temp\ipykernel_9044\2303508014.py:14: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient


API checks:
               check  status
             health     200
   valid_prediction     200
wrong_feature_count     422
Holdout metrics: {
  "accuracy": 0.986013986013986,
  "precision": 0.9888888888888889,
  "recall": 0.9888888888888889,
  "roc_auc": 0.9976939203354298
}
MLflow run: 57080aa212e84156b73b371769ae90fd
Evidence: C:\cynarisis-internship\outputs\w4d4_fastapi_serving
Self-review: model persisted and reloaded; typed endpoints exercised; malformed input rejected; metrics, versions, hash, and run ID recorded.
